In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 
from pathlib import Path

### create masks

#### create "extract-mask" and visualize rows

In [36]:
dir_path = Path('../test/2026-07-23 15-13-06/detail_base_information')
csv_files_path = file_paths = sorted([str(p) for p in dir_path.rglob("*.csv") if p.is_file()])

In [37]:
for i, file_path in enumerate(file_paths):
    print(f"File {str(i).rjust(2, ' ')}: {file_path}")

File  0: ..\test\2026-07-23 15-13-06\detail_base_information\derivatives_bars_daily_futures.csv
File  1: ..\test\2026-07-23 15-13-06\detail_base_information\derivatives_bars_daily_options.csv
File  2: ..\test\2026-07-23 15-13-06\detail_base_information\derivatives_bars_daily_options_225.csv
File  3: ..\test\2026-07-23 15-13-06\detail_base_information\equities_bars_daily.csv
File  4: ..\test\2026-07-23 15-13-06\detail_base_information\equities_earnings-calendar.csv
File  5: ..\test\2026-07-23 15-13-06\detail_base_information\equities_investor-types.csv
File  6: ..\test\2026-07-23 15-13-06\detail_base_information\equities_master.csv
File  7: ..\test\2026-07-23 15-13-06\detail_base_information\fins_details.csv
File  8: ..\test\2026-07-23 15-13-06\detail_base_information\fins_dividend.csv
File  9: ..\test\2026-07-23 15-13-06\detail_base_information\fins_summary.csv
File 10: ..\test\2026-07-23 15-13-06\detail_base_information\indices_bars_daily.csv
File 11: ..\test\2026-07-23 15-13-06\detai

In [40]:
df_calender = pd.read_csv("../test/data/markets_calendar.csv")
df_equities_bars_daily = pd.read_csv(csv_files_path[3])

In [41]:
df_equities_bars_daily['date'] = pd.to_datetime(df_equities_bars_daily['date'])
df_calender['Date'] = pd.to_datetime(df_calender['Date'])

In [42]:
df_equities_bars_daily = df_equities_bars_daily.fillna(-1)

In [ ]:
# please check the start and end day of the data
# if your startdata is not 2008-05-07, please change the start_day value below

df = pd.read_csv(csv_files_path[3])
start_day:str = df[df['date'] == '2008-05-07']['date'].values[0]
end_day:str = df[len(df)-1:]['date'].values[0]
print(f"Start day: {start_day}, End day: {end_day}")

Start day: 2008-05-07, End day: 2026-04-17


In [45]:
from_idx = df_calender[df_calender['Date'] == start_day].index[0]
end_idx  = df_calender[df_calender['Date'] == end_day].index[0]
print(f'from={from_idx}, to={end_idx}, total={end_idx - from_idx + 1}days')

from=127, to=6681, total=6555days


in total 
<span style="color: lightblue; ">6682</span> days, 
<span style="color: lightblue; ">2008-01-01</span> to 
<span style="color: lightblue; ">2026-04-17</span>

In [46]:
plt.figure(figsize=(300, 60))
plt.grid()
plt.plot(df_equities_bars_daily['date'], df_equities_bars_daily['rows'], label='Rows Number', color='blue', lw=10)
plt.xlabel('Date', fontsize=100)
plt.ylabel('Rows Number', fontsize=100)
plt.title('equities_bars_daily of Rows Number Over Time; Raw data', fontsize=300)
plt.legend(fontsize=300)
plt.xticks(rotation=45, fontsize=100)
plt.yticks(fontsize=100)
plt.ylim([-200, 4500])
plt.savefig('../images/raw_data_rows_number_of_equities_bars_daily_over_time.png')
plt.show()

In [47]:
df_equities_bars_daily['date'] = pd.to_datetime(df_equities_bars_daily['date'])
df_calender['Date'] = pd.to_datetime(df_calender['Date'])

In [48]:
df_extract_by_date = df_equities_bars_daily[
    (df_equities_bars_daily['date'] >= start_day) & 
    (df_equities_bars_daily['date'] <= end_day)
]

df_calender_extract_by_date = df_calender[
    (df_calender['Date'] >= start_day) & 
    (df_calender['Date'] <= end_day)
]

assert len(df_extract_by_date) == len(df_calender_extract_by_date)

In [49]:
extract_date_mask = (df_equities_bars_daily['date'] >= start_day) & (df_equities_bars_daily['date'] <= end_day)

assert len(extract_date_mask) == len(df_equities_bars_daily)

np.save('../masks/extract_date_mask.npy', extract_date_mask)

In [50]:
Holiday_mask = (df_calender_extract_by_date['HolDiv'] == 1) | (df_calender_extract_by_date['HolDiv'] == 2)

assert len(df_extract_by_date) == len(Holiday_mask)

In [51]:
df_extract_by_date.index = Holiday_mask.index
masked_df_extract_by_date = df_extract_by_date[Holiday_mask]

In [52]:
masked_df_extract_by_date

,date,name,cols,rows
127,2008-05-07,equities_bars_daily.csv,42.0,2494.0
128,2008-05-08,equities_bars_daily.csv,42.0,2494.0
129,2008-05-09,equities_bars_daily.csv,42.0,2493.0
132,2008-05-12,equities_bars_daily.csv,42.0,2493.0
133,2008-05-13,equities_bars_daily.csv,42.0,2493.0
...,...,...,...,...
6677,2026-04-13,equities_bars_daily.csv,42.0,4449.0
6678,2026-04-14,equities_bars_daily.csv,42.0,4448.0
6679,2026-04-15,equities_bars_daily.csv,42.0,4448.0
6680,2026-04-16,equities_bars_daily.csv,42.0,4448.0


HolDiv（休日区分）が１（営業日）か２（東証半日立会日）の時に分析対象とする

- ０は「非営業日」でありデータが登録されていないため利用しない
- ３は「非営業日(祝日取引あり)」であり，NaNを多く踏むため利用しない

In [53]:
plt.figure(figsize=(300, 60))
plt.grid()
plt.plot(masked_df_extract_by_date['date'], masked_df_extract_by_date['rows'], label='Rows Number', color='blue', lw=10)
plt.xlabel('Date', fontsize=100)
plt.ylabel('Rows Number', fontsize=100)
plt.title('equities_bars_daily of Rows Number Over Time; Limit (Holdiv: [1, 2] Range: [2008-01-01 - 2026-04-17])', fontsize=300)
plt.legend(fontsize=300)
plt.xticks(rotation=45, fontsize=100)
plt.yticks(fontsize=100)
plt.ylim([-200, 4500])
plt.savefig('../images/masked_data_rows_number_of_equities_bars_daily_over_time.png')
plt.show()

In [54]:
cols_and_rows_Nan_mask = ~((df_extract_by_date["cols"] < 0) & (df_extract_by_date["rows"] < 0))
assert len(cols_and_rows_Nan_mask) == len(Holiday_mask)
complete_mask = Holiday_mask & cols_and_rows_Nan_mask

In [55]:
usable_df = df_extract_by_date[complete_mask]

In [56]:
plt.figure(figsize=(300, 60))
plt.grid()
plt.plot(usable_df['date'], usable_df['rows'], label='Rows Number', color='blue', lw=10)
plt.xlabel('Date', fontsize=100)
plt.ylabel('Rows Number', fontsize=100)
plt.title('equities_bars_daily of Rows Number Over Time; Limit (Holdiv: [1, 2] Range: [2008-01-01 - 2026-04-17])', fontsize=300)
plt.legend(fontsize=300)
plt.xticks(rotation=45, fontsize=100)
plt.yticks(fontsize=100)
plt.ylim([-200, 4500])
plt.savefig('../images/usable_data_rows_number_of_equities_bars_daily_over_time.png')
plt.show()

#### completely create "date-mask"
1. HolDiv only use 1 and 2
2. TimeRange only use 2008-01-01 - 2026-04-17 [reference from trade-calender]
3. cols and rows only use non-negative values [data-vendor don't provide data for those days]
4. target-node is stock-node, so use time-range valid-stock-data-range

In [57]:
np.save('../masks/valid_date_mask.npy', complete_mask)

#### create only-date-nparray

In [59]:
extract_date_mask = np.load('../masks/extract_date_mask.npy')
valid_date_mask = np.load('../masks/valid_date_mask.npy')
df_equities_bars_daily = pd.read_csv("../test/2026-07-23 15-13-06/detail_base_information/equities_master.csv")

df_equities_bars_daily = df_equities_bars_daily[extract_date_mask][valid_date_mask].reset_index(drop=True)

In [60]:
df_equities_bars_daily

,date,name,cols,rows
0,2008-05-07,equities_master.csv,14,2494
1,2008-05-08,equities_master.csv,14,2494
2,2008-05-09,equities_master.csv,14,2493
3,2008-05-12,equities_master.csv,14,2493
4,2008-05-13,equities_master.csv,14,2493
...,...,...,...,...
4387,2026-04-13,equities_master.csv,14,4449
4388,2026-04-14,equities_master.csv,14,4448
4389,2026-04-15,equities_master.csv,14,4448
4390,2026-04-16,equities_master.csv,14,4448


In [61]:
only_date_nparray = df_equities_bars_daily['date']

In [62]:
np.save('../masks/only_date_nparray.npy', only_date_nparray)